# 02 - Criterios de Enlace (Linkage Criteria) y Validación Cofenética

## Curso de Machine Learning: Modelos No Supervisados

En el agrupamiento aglomerativo, la distancia entre dos observaciones individuales $x_i, x_j$ se calcula mediante una métrica estándar (Euclidiana, Manhattan, Coseno). Sin embargo, una vez que comenzamos a fusionar puntos en grupos, surge una pregunta matemática crucial: **¿Cómo medimos la distancia entre dos conjuntos de puntos $A$ y $B$?**

Esta regla se denomina **Criterio de Enlace (*Linkage Criterion*)**. La elección del enlace determina qué geometría de clusters es favorecida (esferas compactas, cadenas filamentosas o grupos de igual diámetro) y es el hiperparámetro más influyente del algoritmo.

En este cuaderno estudiaremos en profundidad los cuatro criterios de enlace clásicos, la **Fórmula de Recurrencia de Lance-Williams** y la selección rigurosa del mejor enlace mediante el **Coeficiente de Correlación Cofenética**.

---

### Contenidos
1. Los Cuatro Criterios de Enlace Clásicos (Single, Complete, Average, Ward)
2. Formulación Matemática de Cada Criterio
3. La Fórmula Universal de Recurrencia de Lance-Williams
4. El Efecto Cadena (*Chaining Effect*) en Single Linkage vs. Compacidad en Complete y Ward
5. Comparativa Experimental: Geometría de Formas No Convexas
6. Benchmark de Preservación de Distancias con Correlación Cofenética
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Definición Formal de los Criterios de Enlace

Sean $A$ y $B$ dos clusters disjuntos. La distancia $D(A, B)$ entre ellos se define según el criterio:

### 1. Enlace Simple (*Single Linkage / Mínima Distancia*)
$$D_{single}(A, B) = \min_{a \in A, b \in B} d(a, b)$$
- **Comportamiento**: Busca los dos puntos más cercanos entre ambos clusters. Tiende a formar estructuras alargadas y filamentosas. Puede seguir variedades no convexas (similar a DBSCAN), pero sufre del **efecto cadena (*chaining*)**: si hay un solo punto espurio entre dos clusters densos, el enlace simple los fusionará prematuramente.

### 2. Enlace Completo (*Complete Linkage / Máxima Distancia*)
$$D_{complete}(A, B) = \max_{a \in A, b \in B} d(a, b)$$
- **Comportamiento**: Mide la distancia entre los dos puntos más lejanos (el diámetro del cluster fusionado). Favorece clusters compactos con diámetros máximos similares. Es sumamente sensible a valores atípicos.

### 3. Enlace Promedio (*Average Linkage / UPGMA*)
$$D_{average}(A, B) = \frac{1}{|A| |B|} \sum_{a \in A} \sum_{b \in B} d(a, b)$$
- **Comportamiento**: Promedia todas las distancias cruzadas por pares. Es una solución de compromiso muy robusta frente a ruido y suele maximizar el coeficiente de correlación cofenética.

### 4. Enlace de Ward (*Mínima Varianza / Ward's Minimum Variance*)
$$D_{ward}(A, B) = \frac{|A| |B|}{|A| + |B|} \| \mu_A - \mu_B \|_2^2 = \Delta \text{SSE}_{A \cup B}$$
- **Comportamiento**: En cada paso, fusiona los dos clusters que causen el **menor incremento en la suma de cuadrados de los errores (Inercia / SSE)**. Es el análogo aglomerativo directo de **K-Means**, favoreciendo clusters esféricos y de tamaños equilibrados. **Requiere distancia euclidiana obligatoriamente**.


## 2. La Fórmula de Recurrencia de Lance-Williams (1967)

Calcular desde cero todas las distancias entre clusters después de cada fusión tomaría $O(n^3)$. Lance y Williams demostraron que la distancia entre un cluster recién fusionado $(A \cup B)$ y cualquier otro cluster $C$ puede actualizarse directamente en $O(1)$ sin consultar los puntos originales:

$$D(A \cup B, C) = \alpha_A D(A, C) + \alpha_B D(B, C) + \beta D(A, B) + \gamma |D(A, C) - D(B, C)|$$

| Método de Enlace | $\alpha_A$ | $\alpha_B$ | $\beta$ | $\gamma$ |
| :--- | :---: | :---: | :---: | :---: |
| **Single** | $0.5$ | $0.5$ | $0$ | $-0.5$ |
| **Complete** | $0.5$ | $0.5$ | $0$ | $+0.5$ |
| **Average (UPGMA)** | $\frac{|A|}{|A|+|B|}$ | $\frac{|B|}{|A|+|B|}$ | $0$ | $0$ |
| **Ward** | $\frac{|A|+|C|}{|A|+|B|+|C|}$ | $\frac{|B|+|C|}{|A|+|B|+|C|}$ | $-\frac{|C|}{|A|+|B|+|C|}$ | $0$ |


## 3. Demostración Empírica: Efecto Cadena vs. Compacidad

Comparemos visualmente el comportamiento de los 4 métodos sobre dos variedades no lineales entrelazadas (*Moons*) con ruido.


In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=42)

enlaces = ['single', 'complete', 'average', 'ward']
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, enlace in zip(axes, enlaces):
    agg = AgglomerativeClustering(n_clusters=2, linkage=enlace)
    labels = agg.fit_predict(X_moons)
    ari = adjusted_rand_score(y_moons, labels)
    
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', s=35, edgecolors='k', alpha=0.8)
    ax.set_title(f'Enlace: {enlace.capitalize()}\n(ARI = {ari:.3f})', fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Geometría de Clusters según el Criterio de Enlace en Formas No Convexas', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


## 4. Validación Cofenética sobre un Dataset Multivariado Real

Carguemos un conjunto de datos multivariado continuo y evaluemos cuantitativamente qué método de enlace preserva mejor la geometría del espacio original calculando el **Coeficiente de Correlación Cofenética (CPCC)**.


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine = wine.data
y_wine = wine.target

from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X_wine)
distancias_reales = pdist(X_scaled, metric='euclidean')

tabla_copenetica = []
for enlace in enlaces:
    Z = linkage(X_scaled, method=enlace, metric='euclidean')
    cpcc, _ = cophenet(Z, distancias_reales)
    
    # Evaluamos la pureza de corte en 3 grupos
    agg = AgglomerativeClustering(n_clusters=3, linkage=enlace)
    pred = agg.fit_predict(X_scaled)
    ari = adjusted_rand_score(y_wine, pred)
    
    tabla_copenetica.append({
        'Criterio_Enlace': enlace.capitalize(),
        'Correlacion_Cofenetica_CPCC': round(cpcc, 4),
        'Pureza_Clusters_ARI': round(ari, 4)
    })

df_copenet = pd.DataFrame(tabla_copenetica).sort_values('Correlacion_Cofenetica_CPCC', ascending=False)
print('Ranking de Preservación Métrica (CPCC) y Concordancia (ARI):')
display(df_copenet)


## 5. Visualización Comparativa de los Cuatro Dendrogramas

Observa cómo la topología de los dendrogramas refleja las propiedades teóricas de cada enlace: `single` muestra ramas alargadas y desbalanceadas paso a paso, mientras que `complete` y `ward` forman árboles simétricos y bien jerarquizados.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for idx, enlace in enumerate(enlaces):
    Z = linkage(X_scaled, method=enlace, metric='euclidean')
    dendrogram(Z, ax=axes[idx], p=25, truncate_mode='lastp', show_leaf_counts=True, no_labels=True)
    axes[idx].set_title(f'Dendrograma Truncado: Enlace {enlace.capitalize()} (CPCC = {df_copenet.loc[df_copenet["Criterio_Enlace"] == enlace.capitalize(), "Correlacion_Cofenetica_CPCC"].values[0]:.3f})',
                        fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Distancia de Enlace')
    axes[idx].set_xlabel('Nodos / Hojas Agrupadas')

plt.suptitle('Comparación Visual de la Topología del Dendrograma según el Criterio de Enlace', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Conclusiones

### Reglas Prácticas de Selección de Enlace:
1. **`ward`**: Es la opción predeterminada recomendada para la gran mayoría de problemas de segmentación tabular y perfiles de clientes, ya que produce grupos compactos, balanceados e inmunes a encadenamientos.
2. **`average`**: Excelente cuando el objetivo es reconstruir taxonomías y preservar con máxima fidelidad las distancias métricas originales ($CPCC$).
3. **`complete`**: Ideal si se requiere garantizar que ningún par de observaciones dentro del mismo cluster supere un diámetro máximo estricto.
4. **`single`**: Útil únicamente si se esperan variedades no convexas o se desea detectar anomalías aisladas en la periferia.

En el siguiente cuaderno, **03_Dendrogramas_Cortes_y_Clustermaps.ipynb**, aprenderemos las estrategias óptimas para fijar la línea de corte del árbol y generar mapas de calor jerárquicos bivariados (*Clustermaps*).
